# Environment

In [ ]:
# conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia

In [ ]:
conda list

# train script

In [4]:
import os
import datetime

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.tensorboard import SummaryWriter


'''utils'''

# source: https://github.com/WZMIAOMIAO/deep-learning-for-image-processing
from collections import defaultdict, deque
import datetime
import time
import torch
import torch.distributed as dist
import numpy as np

import errno
import os


class SmoothedValue(object):
    """Track a series of values and provide access to smoothed values over a
    window or the global series average.
    """

    def __init__(self, window_size=20, fmt=None):
        if fmt is None:
            fmt = "{value:.4f} ({global_avg:.4f})"
        self.deque = deque(maxlen=window_size)
        self.total = 0.0
        self.count = 0
        self.fmt = fmt

    def update(self, value, n=1):
        self.deque.append(value)
        self.count += n
        self.total += value * n

    def synchronize_between_processes(self):
        """
        Warning: does not synchronize the deque!
        """
        if not is_dist_avail_and_initialized():
            return
        t = torch.tensor([self.count, self.total], dtype=torch.float64, device='cuda')
        dist.barrier()
        dist.all_reduce(t)
        t = t.tolist()
        self.count = int(t[0])
        self.total = t[1]

    @property
    def median(self):
        d = torch.tensor(list(self.deque))
        return d.median().item()

    @property
    def avg(self):
        d = torch.tensor(list(self.deque), dtype=torch.float32)
        return d.mean().item()

    @property
    def global_avg(self):
        return self.total / self.count

    @property
    def max(self):
        return max(self.deque)

    @property
    def value(self):
        return self.deque[-1]

    def __str__(self):
        return self.fmt.format(
            median=self.median,
            avg=self.avg,
            global_avg=self.global_avg,
            max=self.max,
            value=self.value)


class ConfusionMatrix(object):
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.mat = None

    def update(self, a, b):
        n = self.num_classes
        if self.mat is None:
            # 创建混淆矩阵
            self.mat = torch.zeros((n, n), dtype=torch.int64, device=a.device)
        with torch.no_grad():
            # 寻找GT中为目标的像素索引
            k = (a >= 0) & (a < n)
            # 统计像素真实类别a[k]被预测成类别b[k]的个数(这里的做法很巧妙)
            inds = n * a[k].to(torch.int64) + b[k]
            self.mat += torch.bincount(inds, minlength=n ** 2).reshape(n, n)

    def reset(self):
        if self.mat is not None:
            self.mat.zero_()

    def compute(self):
        h = self.mat.float()
        # 计算全局预测准确率(混淆矩阵的对角线为预测正确的个数)
        acc_global = torch.diag(h).sum() / h.sum()
        # 计算每个类别的准确率
        acc = torch.diag(h) / h.sum(1)
        # 计算每个类别预测与真实目标的iou
        iu = torch.diag(h) / (h.sum(1) + h.sum(0) - torch.diag(h))
        return acc_global, acc, iu

    def reduce_from_all_processes(self):
        if not torch.distributed.is_available():
            return
        if not torch.distributed.is_initialized():
            return
        torch.distributed.barrier()
        torch.distributed.all_reduce(self.mat)

    def __str__(self):
        acc_global, acc, iu = self.compute()
        return (
            'ARC: {}\n'
            'global correct: {:.1f}\n'
            'IoU: {}\n'
            'mean IoU: {:.1f}').format(
            ['{:.1f}'.format(i) for i in (acc * 100).tolist()],
            acc_global.item() * 100,
            ['{:.1f}'.format(i) for i in (iu * 100).tolist()],
            iu.mean().item() * 100)


class MetricLogger(object):
    def __init__(self, delimiter="\t"):
        self.meters = defaultdict(SmoothedValue)
        self.delimiter = delimiter

    def update(self, **kwargs):
        for k, v in kwargs.items():
            if isinstance(v, torch.Tensor):
                v = v.item()
            assert isinstance(v, (float, int))
            self.meters[k].update(v)

    def __getattr__(self, attr):
        if attr in self.meters:
            return self.meters[attr]
        if attr in self.__dict__:
            return self.__dict__[attr]
        raise AttributeError("'{}' object has no attribute '{}'".format(
            type(self).__name__, attr))

    def __str__(self):
        loss_str = []
        for name, meter in self.meters.items():
            loss_str.append(
                "{}: {}".format(name, str(meter))
            )
        return self.delimiter.join(loss_str)

    def synchronize_between_processes(self):
        for meter in self.meters.values():
            meter.synchronize_between_processes()

    def add_meter(self, name, meter):
        self.meters[name] = meter

    def log_every(self, iterable, print_freq, record_mark, header=None):
        i = 0
        if not header:
            header = ''
        start_time = time.time()
        end = time.time()
        iter_time = SmoothedValue(fmt='{avg:.4f}')
        data_time = SmoothedValue(fmt='{avg:.4f}')
        space_fmt = ':' + str(len(str(len(iterable)))) + 'd'
        loggers_file = "work_dir/logger/loggers{}.txt".format(record_mark)
        if torch.cuda.is_available():
            log_msg = self.delimiter.join([
                header,
                '[{0' + space_fmt + '}/{1}]',
                '{now}',
                'eta: {eta}',
                '{meters}',
                'time: {time}',
                'data: {data}',
                'max mem: {memory:.0f}'
            ])
        else:
            log_msg = self.delimiter.join([
                header,
                '[{0' + space_fmt + '}/{1}]',
                '{now}',
                'eta: {eta}',
                '{meters}',
                'time: {time}',
                'data: {data}'
            ])
        MB = 1024.0 * 1024.0
        for obj in iterable:
            data_time.update(time.time() - end)
            yield obj
            iter_time.update(time.time() - end)
            if i % print_freq == 0:
                now = datetime.datetime.now()
                eta_seconds = iter_time.global_avg * (len(iterable) - i)
                eta_time = datetime.timedelta(seconds=int(eta_seconds)) + now
                eta_string = str(datetime.timedelta(seconds=int(eta_seconds))) + '/' + eta_time.strftime("%H:%M:%S")
                if torch.cuda.is_available():
                    print(log_msg.format(
                        i, len(iterable),
                        now=now.strftime("%H:%M:%S"),
                        eta=eta_string,
                        meters=str(self),
                        time=str(iter_time), data=str(data_time),
                        memory=torch.cuda.max_memory_allocated() / MB))
                    with open(loggers_file, "a") as f:
                        f.write(log_msg.format(
                            i, len(iterable),
                            now=datetime.datetime.now().strftime("%H:%M:%S"),
                            eta=eta_string,
                            meters=str(self),
                            time=str(iter_time), data=str(data_time),
                            memory=torch.cuda.max_memory_allocated() / MB) + "\n")
                else:
                    print(log_msg.format(
                        i, len(iterable),
                        now=datetime.datetime.now().strftime("%H:%M:%S"),
                        eta=eta_string,
                        meters=str(self),
                        time=str(iter_time), data=str(data_time)))
                    with open(loggers_file, "a") as f:
                        f.write(log_msg.format(
                            i, len(iterable),
                            now=datetime.datetime.now().strftime("%H:%M:%S"),
                            eta=eta_string,
                            meters=str(self),
                            time=str(iter_time), data=str(data_time)) + "\n")
            i += 1
            end = time.time()
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        with open(loggers_file, "a") as f:
            f.write('{} Total time: {}'.format(header, total_time_str) + "\n")
        print('{} Total time: {}'.format(header, total_time_str))


'''distributed computing setting'''


def mkdir(path):
    try:
        os.makedirs(path)
    except OSError as e:
        if e.errno != errno.EEXIST:
            raise


def setup_for_distributed(is_master):
    """
    This function disables printing when not in master process
    """
    import builtins as __builtin__
    builtin_print = __builtin__.print

    def print(*args, **kwargs):
        force = kwargs.pop('force', False)
        if is_master or force:
            builtin_print(*args, **kwargs)

    __builtin__.print = print


def is_dist_avail_and_initialized():
    if not dist.is_available():
        return False
    if not dist.is_initialized():
        return False
    return True


def get_world_size():
    if not is_dist_avail_and_initialized():
        return 1
    return dist.get_world_size()


def get_rank():
    if not is_dist_avail_and_initialized():
        return 0
    return dist.get_rank()


def is_main_process():
    return get_rank() == 0


def save_on_master(*args, **kwargs):
    if is_main_process():
        torch.save(*args, **kwargs)


def init_distributed_mode(args):
    if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ:
        args.rank = int(os.environ["RANK"])
        args.world_size = int(os.environ['WORLD_SIZE'])
        args.gpu = int(os.environ['LOCAL_RANK'])
    elif 'SLURM_PROCID' in os.environ:
        args.rank = int(os.environ['SLURM_PROCID'])
        args.gpu = args.rank % torch.cuda.device_count()
    elif hasattr(args, "rank"):
        pass
    else:
        print('Not using distributed mode')
        args.distributed = False
        return

    args.distributed = True

    torch.cuda.set_device(args.gpu)
    args.dist_backend = 'nccl'
    print('| distributed init (rank {}): {}'.format(
        args.rank, args.dist_url), flush=True)
    torch.distributed.init_process_group(backend=args.dist_backend, init_method=args.dist_url,
                                         world_size=args.world_size, rank=args.rank)
    setup_for_distributed(args.rank == 0)

'''tra_val'''

# source: https://github.com/WZMIAOMIAO/deep-learning-for-image-processing
import torch
from torch import nn


class FocalLoss():
    def __init__(self, alpha=0.5, gamma=2, weight=None, ignore_index=0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index
        self.ce_fn = nn.CrossEntropyLoss(weight=self.weight, ignore_index=self.ignore_index)

    def __call__(self, inputs, target):
        logpt = -self.ce_fn(inputs, target)
        pt = torch.exp(logpt)
        loss = -self.alpha * ((1 - pt) ** self.gamma) * logpt
        return loss


def criterion(inputs, target):
    # losses = nn.functional.cross_entropy(inputs, target, ignore_index=0)
    fl = FocalLoss(alpha=0.5, gamma=2, weight=None, ignore_index=0)
    losses = fl(inputs, target)
    return losses


def evaluate(model, data_loader, device, num_classes, record_mark):
    model.eval()
    confmat = ConfusionMatrix(num_classes)
    metric_logger = MetricLogger(delimiter="  ")
    header = 'Test:'
    with torch.no_grad():
        for image, annotation, depth in metric_logger.log_every(data_loader, 100, record_mark, header):
            image, annotation, depth = image.to(device), annotation.to(device), depth.to(device)
            output = model(image, depth)

            confmat.update(annotation.flatten(), output.argmax(1).flatten())

        confmat.reduce_from_all_processes()

    return confmat


def create_lr_scheduler(optimizer, half_life, lr_min):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=half_life, eta_min=lr_min)


def train_one_epoch(model, optimizer, data_loader, device, epoch, writer, lr_scheduler, record_mark, print_freq=10,
                    scaler=None):
    """
    Args:
        model: model to be trained
        optimizer: learning rate setting
        data_loader: dataset
        device: device training on
        epoch: epoch times
        lr_scheduler: warm up
        print_freq: report time
        record_mark: record file mark
        scaler: Automatic Mixed Precision
    """
    model.train()
    metric_logger = MetricLogger(delimiter="  ")
    metric_logger.add_meter('lr', SmoothedValue(window_size=1, fmt='{value:.4e}'))
    header = 'Epoch: [{}]'.format(epoch)

    step = 0
    for image, annotation, depth in metric_logger.log_every(data_loader, print_freq, record_mark, header):
        image, annotation, depth = image.to(device), annotation.to(device), depth.to(device)
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            output = model(image, depth)
            loss = criterion(output, annotation)
            if record_mark is not None:
                writer.add_scalar("loss/batch", loss, step + (len(data_loader) + 10) * epoch)

        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        # lr_scheduler.step()

        lr = optimizer.param_groups[0]["lr"]
        metric_logger.update(loss=loss.item(), lr=lr)
        if record_mark is not None:
            writer.add_scalar("lr/batch", lr, step + (len(data_loader) + 10) * epoch)
        step += 1

    lr_scheduler.step()

    return metric_logger.meters["loss"].global_avg, lr


'''dataset'''
import os
import json

import numpy as np
import torch.utils.data as data
from PIL import Image

CLASSES = ['<UNK>', 'beam', 'board', 'bookcase', 'ceiling', 'chair', 'clutter', 'column', 'door', 'floor', 'sofa',
           'table', 'wall', 'window']


class Stanford2D3D(data.Dataset):
    def __init__(self, data_path, flod_num, pipeline, transforms=None):
        super().__init__()
        fold = {"training": [[1, 2, 3, 4, 6], [1, 3, 4, 6], [2, 4, 5], [4]],
                "validation": [[5], [2, 4], [1, 3, 6], [3]]}
        area = ["area_1", "area_2", "area_3", "area_4", "area_5", "area_6"]
        types = ["rgb", "semantic", "depth"]

        self.images = []
        self.annotations = []
        self.depth = []
        for f in fold[pipeline][flod_num]:
            for t in types:
                image_dir = os.path.join(data_path, area[f - 1], t)
                assert os.path.exists(image_dir), "path '{}' does not exist.".format(image_dir)
                for _, _, p in os.walk(image_dir):
                    if t == "rgb":
                        self.images.extend([os.path.join(image_dir, x) for x in p])
                    elif t == "semantic":
                        self.annotations.extend([os.path.join(image_dir, x) for x in p])
                    elif t == "depth":
                        self.depth.extend([os.path.join(image_dir, x) for x in p])
        assert (len(self.images) == len(self.annotations))
        assert (len(self.images) == len(self.depth))

        self.transforms = transforms

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        """
        Args:
            index (int): Index

        Returns:
            tuple: (image, annotation, depth)
            image: RGB
            annotation: semantic
            depth: depth map
        """
        assert self.images[index].split(os.sep)[3][:-7] == self.annotations[index].split(os.sep)[3][:-12], \
            "RGB {} doesn't match annotation {}.".format(self.images[index], self.annotations[index])
        assert self.images[index].split(os.sep)[3][:-7] == self.depth[index].split(os.sep)[3][:-9], \
            "RGB {} doesn't match depth {}.".format(self.images[index], self.annotations[index])
        image = Image.open(self.images[index])
        annotation = Image.open(self.annotations[index])
        depth = Image.open(self.depth[index])

        if self.transforms is not None:
            image, annotation, depth = self.transforms(image, annotation, depth)

        return image, annotation, depth

    @staticmethod
    def collate_fn(batch):
        images, annotations, depth = list(zip(*batch))
        batched_images = cat_list(images, fill_value=0)
        batched_annotations = cat_list(annotations, fill_value=0)
        batched_depth = cat_list(depth, fill_value=65535)
        return batched_images, batched_annotations, batched_depth


def cat_list(images, fill_value=0):
    # 计算该batch数据中，channel, h, w的最大值
    max_size = tuple(max(s) for s in zip(*[img.shape for img in images]))
    batch_shape = (len(images),) + max_size
    batched_imgs = images[0].new(*batch_shape).fill_(fill_value)
    for img, pad_img in zip(images, batched_imgs):
        pad_img[..., :img.shape[-2], :img.shape[-1]].copy_(img)
    return batched_imgs


'''data transform'''
import numpy as np
import random

import torch
from torchvision import transforms as T
from torchvision.transforms import functional as F


def pad_if_smaller(img, size, fill=0):
    min_size = min(img.size)
    if min_size < size:
        ow, oh = img.size
        padh = size - oh if oh < size else 0
        padw = size - ow if ow < size else 0
        img = F.pad(img, [0, 0, padw, padh], fill=fill)
    return img


class Compose(object):
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, image, annotation, depth):
        for t in self.transforms:
            image, annotation, depth = t(image, annotation, depth)
        return image, annotation, depth


class RandomResize(object):
    def __init__(self, min_size, max_size=None):
        self.min_size = min_size
        if max_size is None:
            max_size = min_size
        self.max_size = max_size

    def __call__(self, image, annotation, depth):
        size = random.randint(self.min_size, self.max_size)
        # 这里size传入的是int类型，所以是将图像的最小边长缩放到size大小
        image = F.resize(image, size)
        # 这里的interpolation注意下，在torchvision(0.9.0)以后才有InterpolationMode.NEAREST
        # 如果是之前的版本需要使用PIL.Image.NEAREST
        annotation = F.resize(annotation, size, interpolation=T.InterpolationMode.NEAREST)
        depth = F.resize(depth, size, interpolation=T.InterpolationMode.NEAREST)
        return image, annotation, depth


class RandomHorizontalFlip(object):
    def __init__(self, flip_prob):
        self.flip_prob = flip_prob

    def __call__(self, image, annotation, depth):
        if random.random() < self.flip_prob:
            image = F.hflip(image)
            annotation = F.hflip(annotation)
            depth = F.hflip(depth)
        return image, annotation, depth


class RandomCrop(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, image, annotation, depth):
        image = pad_if_smaller(image, self.size)
        annotation = pad_if_smaller(annotation, self.size, fill=0)
        depth = pad_if_smaller(depth, self.size, fill=65536)
        crop_params = T.RandomCrop.get_params(image, (self.size, self.size))
        image = F.crop(image, *crop_params)
        annotation = F.crop(annotation, *crop_params)
        depth = F.crop(depth, *crop_params)
        return image, annotation, depth


class CenterCrop(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, image, annotation, depth):
        image = F.center_crop(image, self.size)
        annotation = F.center_crop(annotation, self.size)
        depth = F.center_crop(depth, self.size)
        return image, annotation, depth


class ToTensor(object):
    def __call__(self, image, annotation, depth):
        image = F.to_tensor(image)
        annotation = torch.as_tensor(np.array(annotation), dtype=torch.int64)
        depth = torch.as_tensor(np.array(depth), dtype=torch.int64)
        return image, annotation, depth


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, image, annotation, depth):
        image = F.normalize(image, mean=self.mean, std=self.std)
        return image, annotation, depth

    
'''model'''

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channel, out_channel, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=out_channel,
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channel)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(in_channels=out_channel, out_channels=out_channel,
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channel)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    """
    注意：原论文中，在虚线残差结构的主分支上，第一个1x1卷积层的步距是2，第二个3x3卷积层步距是1。
    但在pytorch官方实现过程中是第一个1x1卷积层的步距是1，第二个3x3卷积层步距是2，
    这么做的好处是能够在top1上提升大概0.5%的准确率。
    可参考Resnet v1.5 https://ngc.nvidia.com/catalog/model-scripts/nvidia:resnet_50_v1_5_for_pytorch
    """
    expansion = 4

    def __init__(self, in_channel, out_channel, stride=1, downsample=None):
        super(Bottleneck, self).__init__()

        width = int(out_channel / self.expansion)

        self.conv1 = nn.Conv2d(in_channels=in_channel, out_channels=width,
                               kernel_size=1, stride=1, bias=False)  # squeeze channels
        self.bn1 = nn.BatchNorm2d(width)
        # -----------------------------------------
        self.conv2 = nn.Conv2d(in_channels=width, out_channels=width,
                               kernel_size=3, stride=stride, bias=False, padding=1)
        self.bn2 = nn.BatchNorm2d(width)
        # -----------------------------------------
        self.conv3 = nn.Conv2d(in_channels=width, out_channels=width * self.expansion,
                               kernel_size=1, stride=1, bias=False)  # unsqueeze channels
        self.bn3 = nn.BatchNorm2d(width * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)

        return out


class Fusion(nn.Module):
    def __init__(self, in_channel):
        super().__init__()

        self.in_conv = nn.Sequential(
            nn.Conv2d(in_channel, int(in_channel / 2), kernel_size=1, bias=False),
            nn.BatchNorm2d(int(in_channel / 2))
        )

        self.mix_conv = nn.Sequential(Bottleneck(in_channel, in_channel))

        self.resize = nn.Sequential(
            nn.Conv2d(in_channel, in_channel, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(in_channel)
        )

    def forward(self, rgb, depth):
        fusion = torch.cat((self.in_conv(rgb), self.in_conv(depth)), dim=1)
        fusion = self.mix_conv(fusion)

        return self.resize(fusion), self.resize(fusion + rgb), self.resize(fusion + depth)


class SDCombo(nn.Module):
    def __init__(self, class_num=14, mark=None):
        super().__init__()
        block = [3, 3, 5, 2]
        channels = [64, 128, 256, 512]

        self.in_conv_rgb = nn.Sequential(
            nn.Conv2d(3, channels[0], kernel_size=1, bias=False),
            nn.BatchNorm2d(channels[0])
        )
        self.in_conv_depth = nn.Sequential(
            nn.Conv2d(1, channels[0], kernel_size=1, bias=False),
            nn.BatchNorm2d(channels[0])
        )

        res_block = []
        self.res_rgb = nn.ModuleList()
        self.res_depth = nn.ModuleList()
        for b in range(block[0]):
            res_block.append(BasicBlock(channels[0], channels[0]))
        self.res_rgb.append(nn.Sequential(*res_block))
        self.res_depth.append(nn.Sequential(*res_block))
        res_block.clear()

        fusion = [Fusion(channels[0])]
        for i in range(len(channels) - 1):
            downsample = nn.Sequential(
                nn.Conv2d(channels[i], channels[i + 1], kernel_size=1),
                nn.BatchNorm2d(channels[i + 1])
            )
            for b in range(len(block) - 1):
                res_block.append(BasicBlock(channels[i + 1], channels[i + 1]))
            self.res_rgb.append(nn.Sequential(Bottleneck(channels[i], channels[i + 1], downsample=downsample),
                                              nn.Sequential(*res_block)))
            self.res_depth.append(nn.Sequential(Bottleneck(channels[i], channels[i + 1], downsample=downsample),
                                                nn.Sequential(*res_block)))
            res_block.clear()
            fusion.append(Fusion(channels[i + 1]))

        self.fusion = nn.ModuleList(fusion)

        self.in_conv = []
        for i in range(len(channels) - 1):
            self.in_conv.append(nn.Sequential(
                nn.Conv2d(channels[i + 1], channels[i], kernel_size=1, bias=False),
                nn.BatchNorm2d(channels[i])
            ))
        self.in_conv = nn.ModuleList(self.in_conv)

        self.out_conv = nn.Sequential(
            nn.Conv2d(channels[0], channels[0], kernel_size=3, stride=1, padding=1, bias=False),
            nn.SyncBatchNorm(channels[0]),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels[0], class_num, kernel_size=1, bias=True)
        )

    def forward(self, rgb, depth):
        input_size = rgb.size()[-2:]
        rgb = self.in_conv_rgb(rgb)
        depth = F.normalize(depth.type(torch.float))
        depth = self.in_conv_depth(depth.unsqueeze(1))

        mix = []
        for i in range(len(self.fusion)):
            rgb = self.res_rgb[i](rgb)
            depth = self.res_depth[i](depth)
            m, rgb, depth = self.fusion[i](rgb, depth)
            mix.append(m)

        f = mix[-1]
        for i in reversed(range(len(mix) - 1)):
            conv_x = self.in_conv[i](f)
            conv_x = F.interpolate(conv_x, mix[i].size()[-2:], mode='bilinear', align_corners=False)
            f = conv_x + mix[i]

        f = F.interpolate(f, input_size, mode='bilinear', align_corners=False)
        output = self.out_conv(f)
        return output

'''training script'''

def create_model(pretrained, num_classes, mark = None):
    model = SDCombo(num_classes, mark)
    missing_keys = []
    unexpected_keys = []
    if pretrained is not None:
        weights_dict = torch.load(pretrained, map_location='cpu')['model']
        missing_keys, unexpected_keys = model.load_state_dict(weights_dict, strict=False)
        if len(missing_keys) != 0:
            print("missing keys: ", end='')
            for i in missing_keys:
                print(i, end=', ')
            print('\n')
        if len(unexpected_keys) != 0:
            print("unexpected_keys: ", end='')
            for i in unexpected_keys:
                print(i, end=', ')
            print('\n')
    return model, missing_keys, unexpected_keys


def get_transform(train):
    base_size = 1080
    crop_size = 256
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)
    return PipelineTrain(base_size, crop_size, mean=mean, std=std) if train else PipelineEval(crop_size, mean=mean,
                                                                                              std=std)


class PipelineTrain:
    def __init__(self, base_size, crop_size, mean, std, hflip_prob=0.5):
        min_size = int(0.25 * base_size)
        max_size = int(0.5 * base_size)

        trans = [RandomResize(min_size, max_size)]
        if hflip_prob > 0:
            trans.append(RandomHorizontalFlip(hflip_prob))
        trans.extend([
            RandomCrop(crop_size),
            ToTensor(),
            Normalize(mean=mean, std=std)
        ])
        self.transforms = Compose(trans)

    def __call__(self, image, annotation, depth):
        return self.transforms(image, annotation, depth)


class PipelineEval:
    def __init__(self, crop_size, mean, std):
        self.transforms = Compose([
            RandomCrop(crop_size),
            ToTensor(),
            Normalize(mean=mean, std=std),
        ])

    def __call__(self, image, annotation, depth):
        return self.transforms(image, annotation, depth)


def main(args):
    torch.cuda.empty_cache()
    # device, batch size, number of classes and workers
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    batch_size = args.batch_size
    num_classes = args.num_classes
    num_workers = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])
    # record mark
    if args.test:
        mark = None
    else:
        mark = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # build dataset
    train_dataset = Stanford2D3D(args.data_path, args.fold_num - 1, "training", transforms=get_transform(train=True))
    val_dataset = Stanford2D3D(args.data_path, args.fold_num - 1, "validation", transforms=get_transform(train=False))
    # train_dataset = VKITTI(args.data_path, "training", transforms=get_transform(train=True))
    # val_dataset = VKITTI(args.data_path, "validation", transforms=get_transform(train=False))

    train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=batch_size,
                                               num_workers=num_workers,
                                               shuffle=True,
                                               pin_memory=True,
                                               collate_fn=train_dataset.collate_fn)

    val_loader = torch.utils.data.DataLoader(val_dataset,
                                             batch_size=1,
                                             num_workers=num_workers,
                                             pin_memory=True,
                                             collate_fn=val_dataset.collate_fn)

    # build the model from pretrained and local it on device
    pretrained = args.pretrained
    model, missing_keys, unexpected_keys = create_model(pretrained, num_classes, mark)
    model.to(device)

    # collect the parameters to be optimized
    if args.pretrained is not None:
        for n, p in model.named_parameters():
            if n in missing_keys:
                p.requires_grad = True
            elif n.split('.')[0] in args.module_trained and n not in missing_keys:
                p.requires_grad = True
            else:
                p.requires_grad = False

    params_to_optimize = [p for p in model.parameters() if p.requires_grad]
    print("Parameters unfreeze:")
    for i in [n for n, p in model.named_parameters() if p.requires_grad]:
        print(i)

    # set the optimizer
    # use AdamW
    optimizer = torch.optim.AdamW(
        params_to_optimize,
        lr=args.lr,
        betas=(0.9, 0.999),
        weight_decay=args.weight_decay
    )
    scaler = torch.cuda.amp.GradScaler() if args.amp else None
    # update each iteration by CosineAnnealingLR
    lr_scheduler = create_lr_scheduler(optimizer, args.cos[0], args.cos[1])

    # resume
    if args.resume:
        checkpoint = torch.load(args.resume, map_location='cpu')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
        args.start_epoch = checkpoint['epoch'] + 1
        # if args.amp:
        #     scaler.load_state_dict(checkpoint["scaler"])
        _,file = os.path.split(args.resume)
        mark = file[6:21]

    # record file
    if mark is not None:
        results_file = "work_dir/evaluation/evaluation{}.txt".format(mark)
        # log meta info
        args_dict = vars(args)
        train_meta = [i + ": " + str(args_dict[i]) + '\n' for i in args_dict.keys()]
        train_meta_str = ""
        for s in train_meta:
            train_meta_str += s
        train_meta_str += "mark: " + mark +'\n'
        with (open(results_file, "a") as f):
            f.write(train_meta_str + "\n")

    # start the training
    start_time = datetime.datetime.now()
    if mark is not None:
        writer = SummaryWriter("work_dir/board/"+mark)
    else:
        writer = None
    for epoch in range(args.start_epoch, args.epochs):
        # loss and learning rate
        mean_loss, lr = train_one_epoch(model, optimizer, train_loader, device, epoch, writer, lr_scheduler, mark,
                                        args.print_freq, scaler)
        if mark is not None:
            writer.add_scalar("loss/epoch", mean_loss, epoch)
            writer.add_scalar("lr/epoch", lr, epoch)
        torch.cuda.empty_cache()
        confmat = evaluate(model, val_loader, device=device, num_classes=num_classes, record_mark=mark)
        val_info = str(confmat)
        print(val_info)
        if mark is not None:
            writer.add_scalar("Acc/epoch", float(val_info.split('\n')[2].split(':')[1]), epoch)
            writer.add_scalar("mIoU/epoch", float(val_info.split('\n')[4].split(':')[1]), epoch)
            iou = [float(i) for i in val_info.split('\n')[3].split(':')[1][2:-1].replace("'",'').split(',')]
            cls = val_info.split('\n')[0].split(':')[1][2:-2].replace("'", '').split(',')
            for i in range(len(iou)):
                writer.add_scalar("IoU/" + cls[i], iou[i], epoch)
            # write into txt
            with open(results_file, "a") as f:
                # record: train_loss, lr, val_set corresponding to each epoch
                train_info = f"[epoch: {epoch}]\n" \
                             f"time: {datetime.datetime.now().strftime('%H:%M:%S')}\n" \
                             f"train_loss: {mean_loss:.4f}\n" \
                             f"lr: {lr:.4e}\n"
                f.write(train_info + val_info + "\n\n")

            save_file = {"model": model.state_dict(),
                         "optimizer": optimizer.state_dict(),
                         "lr_scheduler": lr_scheduler.state_dict(),
                         "epoch": epoch,
                         "args": args}
            if args.amp:
                save_file["scaler"] = scaler.state_dict()
            torch.save(save_file, "work_dir/model/model_{}_{}.pth".format(mark, epoch))

    total_time = datetime.datetime.now() - start_time
    if mark is not None:
        with open(results_file, "a") as f: f.write("training time {}".format(total_time) + "\n\n")
    print("training time {}".format(total_time))


def parse_args():
    import argparse
    parser = argparse.ArgumentParser(description="SDCombo training")

    parser.add_argument("--test", default=False, type=bool, help="Test")
    parser.add_argument("--meaasge", default="ResNet block = [3, 3, 5, 2]", type=str, help="Message")

    parser.add_argument("--data-path", default="/kaggle/input/Stanford2D3D", help="Dataset root")
    parser.add_argument("--fold-num", default=4, type=int,
                        help="Training & Testing allocation:\n"
                             "1: [[1, 2, 3, 4, 6], [5]],\n2: [[1, 3, 5, 6], [2, 4]],\n"
                             "3: [[2, 4, 5], [1, 3, 6]],\n4: [[4], [3]]\n5: [[7], [8]]\n"
                             "Data size: [10327, 15714, 3704, 13268, 17593, 9890]"
                             "1: [52903, 17593], 2: [41514, 28982], 3: [46575, 23921], 4: [13268, 3704]")
    parser.add_argument("--num-classes", default=14, type=int)
    parser.add_argument("-b", "--batch-size", default=4, type=int)

    parser.add_argument("--aux", default=True, type=bool, help="auxiliary loss")
    parser.add_argument("--device", default="cuda", help="training device")
    parser.add_argument("--epochs", default=20, type=int, metavar="N",
                        help="number of total epochs to train")
    parser.add_argument('--lr', default=5e-3, type=float, help='initial learning rate')
    parser.add_argument('--cos', default=[19, 1e-7], type=list,
                        help='[half-life epoch, minimum learning rate]')
    parser.add_argument('--wd', '--weight-decay', default=0.05, type=float,
                        metavar='W', help='weight decay (default: 0.05)',
                        dest='weight_decay')

    parser.add_argument('--print-freq', default=100, type=int, help='print frequency')
    parser.add_argument('--resume', default='', help='resume from checkpoint')
    parser.add_argument('--start-epoch', default=0, type=int, metavar='N',
                        help='start epoch')
    # Mixed precision training parameters
    parser.add_argument("--amp", default=True, type=bool,
                        help="Use torch.cuda.amp for mixed precision training")
    parser.add_argument('--module_trained',
                        default=['internimage', 'SDHead', 'SDBottleneck'],
                        help='module to be trained: internimage, upernet, SDHead, SDBottleneck')
    parser.add_argument("--pretrained", default=None,
                        help="Pretrained weight, work_dir/model/init_from_ade20k_internimage.pth")

    args = parser.parse_args()

    return args


if __name__ == '__main__':
    args = parse_args()

    if not os.path.exists("work_dir"):
        os.mkdir("work_dir")

    main(args)


usage: ipykernel_launcher.py [-h] [--test TEST] [--meaasge MEAASGE] [--data-path DATA_PATH] [--fold-num FOLD_NUM]
                             [--num-classes NUM_CLASSES] [-b BATCH_SIZE] [--aux AUX] [--device DEVICE] [--epochs N]
                             [--lr LR] [--cos COS] [--wd W] [--print-freq PRINT_FREQ] [--resume RESUME]
                             [--start-epoch N] [--amp AMP] [--module_trained MODULE_TRAINED] [--pretrained PRETRAINED]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\noval\AppData\Roaming\jupyter\runtime\kernel-82afe5d2-d613-4798-9266-6ad21ba487e3.json


SystemExit: 2